# Toxicity Detection & Social Media Analysis

## Notebook 06 – Bluesky Toxicity & Bias Inference 

### Objectives

- Apply the finalized gender/race keyword patterns (whichever list notebook 05 actually decided
  on) to Bluesky posts, independent of the model.
- Run the trained DeBERTa-v3-base model on Bluesky text to get toxicity scores.
- Compare mean toxicity **with** keyword-based grouping against the **overall, ungrouped**
  baseline — i.e. what looking at gender/race keyword flags changes about the toxicity picture,
  versus not looking at them at all.
- Process at scale with chunking + checkpointed shards, so an interruption doesn't lose completed
  work.
- Save a clean, schema-matched deliverable for the visualization-only notebook that follows this
  one.




### Sample mode

The full dataset is **15,223,017 rows**. At the batch size used here that's a multi-hour job on
Kaggle's GPU quota, not a "run all cells" job. `SAMPLE_SIZE` (default `200_000`) makes every
downstream step run on a random subset and finish in well under an hour, using the *exact same
code path* the full run uses. Set `SAMPLE_SIZE = None` once the sample output looks right and
you're ready to commit to the full run; sample and full runs write to separate shard
directories, so switching modes never mixes or overwrites the other run's checkpoints.


In [31]:
from pathlib import Path
import re
import sys
import subprocess
import time
import math

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import precision_recall_fscore_support, classification_report

from tqdm.auto import tqdm
tqdm.pandas()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)


def ensure_package(import_name, pip_name=None):
    '''Best-effort import-or-install. Returns True if the package is usable, False otherwise.
    Used for optional dependencies (language detection, text-cleaning verification) so a missing
    package degrades that one step gracefully instead of crashing the whole notebook.'''
    try:
        __import__(import_name)
        return True
    except ImportError:
        pip_name = pip_name or import_name
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", pip_name],
                check=True, capture_output=True, timeout=180,
            )
            __import__(import_name)
            return True
        except Exception as e:
            print(f"  (optional) could not install '{pip_name}': {e}")
            return False


print("Imports OK.")


Imports OK.


## Paths

Same dataset root as notebook 05. Two new inputs vs. notebook 05: notebook 05's own saved output
(now re-uploaded as part of the dataset) and the processed Bluesky data.


In [32]:
ROOT = Path("..")

# Notebook 05's saved output.
VALIDATION_PATH = ROOT / "models" / "deberta_v3_base" / "predictions" / "jigsaw_gender_race_validation.parquet"

# Already-processed Bluesky data (materialized parquet, not the raw tar).
SOCIAL_PROCESSED_PATH = ROOT / "Dataset" / "processed" / "social_processed.parquet"

# Trained model (same checkpoint as notebook 05)
MODEL_DIR = ROOT / "models" / "deberta_v3_base" / "best_model"

# Keyword lists (same files as Notebook 05)
GENDER_KEYWORDS_PATH = ROOT / "Dataset" / "gender.txt"
RACE_KEYWORDS_PATH = ROOT / "Dataset" / "race.txt"

MAX_LENGTH = 384
INFERENCE_BATCH_SIZE = 64
CHUNK_SIZE = 50_000  # rows per checkpointed shard

# ------------------------------------------------------------------
# SAMPLE MODE -- run on a random subset first to verify the whole pipeline
# end-to-end before committing to the full 15.2M-row run.
#   - Bumped from 20_000 to 200_000: the smallest group bucket (gender_and_race) had only
#     ~35 posts at 20K, too thin to trust. 200K gives ~10x the counts in every bucket while
#     still finishing well under an hour.
#   - Set SAMPLE_SIZE = None to process the full dataset.
#   - Sample and full runs write to DIFFERENT shard dirs / output files (via RUN_TAG below),
#     so switching modes never mixes or overwrites the other run's checkpoints.
# ------------------------------------------------------------------
SAMPLE_SIZE = 200_000
RANDOM_STATE = 42

# Language detection adds real time (pure-Python, roughly 1-2K posts/sec/core). It's checkpointed
# alongside inference so it survives interruptions, but you can turn it off here to save time --
# the rest of the pipeline still runs, you just won't get the English-only comparison.
RUN_LANGUAGE_DETECTION = True

# ------------------------------------------------------------------
# Calibration constants -- see "Prior-corrected toxicity probability" section below.
# TRAIN_POS_RATE: the class balance the model was actually trained on (confirmed in notebooks
#   03 & 04: 40,000/40,000 train, 10,000/10,000 valid -- an exact 50/50 split).
# TRUE_TOXIC_RATE: Jigsaw's real, un-resampled toxic rate (confirmed in notebook 01:
#   train_df["is_toxic"].value_counts(normalize=True) -> 7.9969% positive).
# ------------------------------------------------------------------
TRAIN_POS_RATE = 0.5
TRUE_TOXIC_RATE = 0.079969

RUN_TAG = f"sample{SAMPLE_SIZE}" if SAMPLE_SIZE is not None else "full"

# Output paths
WORKING_DIR = ROOT / "models" / "deberta_v3_base" / "predictions"
SHARD_DIR = WORKING_DIR / "bluesky_shards"
FINAL_OUTPUT_PATH = WORKING_DIR / f"bluesky_toxicity_bias_inference_{RUN_TAG}.parquet"

SHARD_DIR.mkdir(parents=True, exist_ok=True)

print("Validation (nb05 output) :", VALIDATION_PATH, VALIDATION_PATH.exists())
print("Social processed (Bluesky):", SOCIAL_PROCESSED_PATH, SOCIAL_PROCESSED_PATH.exists())
print("Model dir                :", MODEL_DIR, MODEL_DIR.exists())
print("Gender keywords          :", GENDER_KEYWORDS_PATH, GENDER_KEYWORDS_PATH.exists())
print("Race keywords            :", RACE_KEYWORDS_PATH, RACE_KEYWORDS_PATH.exists())
print()
print(f"RUN MODE: {'SAMPLE (' + format(SAMPLE_SIZE, ',') + ' rows)' if SAMPLE_SIZE else 'FULL DATASET'}")
print("Language detection       :", "ON" if RUN_LANGUAGE_DETECTION else "OFF")
print("Shard checkpoint dir     :", SHARD_DIR)
print("Final output path        :", FINAL_OUTPUT_PATH)


Validation (nb05 output) : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/jigsaw_gender_race_validation.parquet True
Social processed (Bluesky): /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/social_processed.parquet True
Model dir                : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/deberta_v3_base/deberta_v3_base/best_model True
Gender keywords          : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/gender.txt True
Race keywords            : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/race.txt True

RUN MODE: SAMPLE (200,000 rows)
Language detection       : ON
Shard checkpoint dir     : /kaggle/working/bluesky_shards_sample200000
Final output path        : /kaggle/working/bluesky_toxicity_bias_inference_sample200000.parquet


In [33]:
validation_df = pd.read_parquet(VALIDATION_PATH)
print("Notebook 05 validation output shape:", validation_df.shape)
print("Columns:", validation_df.columns.tolist())

TRIMMED_LIST_USED = "gender_related_kw_full" in validation_df.columns

if TRIMMED_LIST_USED:
    print("\n--> DETECTED: notebook 05 adopted the TRIMMED (pronoun-dropped) gender keyword list.")
else:
    print("\n--> DETECTED: notebook 05 kept the ORIGINAL (full) gender keyword list.")
    print("    (trimmed variant didn't clear the 2x-precision / 0.75-recall-floor bar)")

# Recompute precision/recall on the saved validation set purely for transparency in this
# notebook's output -- not used to make the decision, just to show the numbers behind it.
def evaluate_keyword_method(y_true, y_pred, label):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    print(f"{label}: precision={precision:.4f}  recall={recall:.4f}  f1={f1:.4f}")
    return precision, recall, f1

print()
final_gender_precision, final_gender_recall, _ = evaluate_keyword_method(
    validation_df["gender_related_annot"], validation_df["gender_related_kw"],
    "Gender keyword list actually in effect (gender_related_kw)"
)
if TRIMMED_LIST_USED:
    evaluate_keyword_method(
        validation_df["gender_related_annot"], validation_df["gender_related_kw_full"],
        "Original list, kept for reference (gender_related_kw_full)"
    )
_ = evaluate_keyword_method(
    validation_df["race_related_annot"], validation_df["race_related_kw"],
    "Race keyword list (unchanged, full list)"
)


Notebook 05 validation output shape: (19987, 21)
Columns: ['id', 'comment_text', 'clean_text', 'target', 'is_toxic', 'gender_related_annot', 'race_related_annot', 'gender_related_kw', 'gender_related_kw_full', 'race_related_kw', 'pred_prob', 'pred_label', 'female', 'male', 'transgender', 'other_gender', 'black', 'white', 'asian', 'latino', 'other_race_or_ethnicity']

--> DETECTED: notebook 05 adopted the TRIMMED (pronoun-dropped) gender keyword list.

Gender keyword list actually in effect (gender_related_kw): precision=0.4942  recall=0.9563  f1=0.6516
Original list, kept for reference (gender_related_kw_full): precision=0.1234  recall=0.9742  f1=0.2190
Race keyword list (unchanged, full list): precision=0.4163  recall=0.9239  f1=0.5740


In [34]:
def load_keywords(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip().lower() for line in f]
    return [line for line in lines if line]


def build_pattern(keywords):
    escaped = [re.escape(kw) for kw in keywords]
    pattern = r"\b(?:" + "|".join(escaped) + r")\b"
    return re.compile(pattern, flags=re.IGNORECASE)


PRONOUNS_TO_DROP = ["he", "him", "his", "she", "her", "hers", "they", "them", "theirs"]

gender_keywords_raw = load_keywords(GENDER_KEYWORDS_PATH)
race_keywords = load_keywords(RACE_KEYWORDS_PATH)

if TRIMMED_LIST_USED:
    gender_keywords_final = [kw for kw in gender_keywords_raw if kw not in PRONOUNS_TO_DROP]
else:
    gender_keywords_final = gender_keywords_raw

print(f"Gender keyword list in effect for Bluesky: {len(gender_keywords_final)} terms "
      f"({'trimmed' if TRIMMED_LIST_USED else 'original'})")
print(f"Race keyword list in effect for Bluesky: {len(race_keywords)} terms (unchanged, per nb05)")

gender_pattern = build_pattern(gender_keywords_final)
race_pattern = build_pattern(race_keywords)

print("\nPatterns built.")


Gender keyword list in effect for Bluesky: 28 terms (trimmed)
Race keyword list in effect for Bluesky: 40 terms (unchanged, per nb05)

Patterns built.


## Load Bluesky Data

Reading the already-processed parquet directly (see environment note above -- no tar streaming
needed here).


In [35]:
t0 = time.time()
bluesky = pd.read_parquet(SOCIAL_PROCESSED_PATH)
print(f"Loaded {SOCIAL_PROCESSED_PATH.name} in {time.time() - t0:.1f}s")
print("Shape (full dataset):", bluesky.shape)
print("Columns:", bluesky.columns.tolist())
bluesky.head(3)


Loaded social_processed.parquet in 24.2s
Shape (full dataset): (15223017, 8)
Columns: ['post_id', 'user_id', 'instance', 'date', 'text', 'sent_label', 'sent_score', 'clean_text']


,post_id,user_id,instance,date,text,sent_label,sent_score,clean_text
0,147967305,1000000,bsky.social,1976-05-31 15:06:22.239,おはようございます！,NaN,NaN,!
1,147967307,1000000,bsky.social,1976-05-31 15:05:51.253,気になります。食べてみたい。,NaN,NaN,
2,147967319,1000000,bsky.social,1976-05-31 14:49:41.210,需要が高いのはお写真ですかね。Xに上げてない過去のとか、Xを補完するようなオフショット的なものとか。,NaN,NaN,xx


### Applying sample mode (if enabled)

Everything from here on -- text-cleaning verification, duplicate check, keyword matching,
chunked inference, language detection, group comparison, final save -- runs on whatever
`bluesky` points to. In sample mode that's a random `SAMPLE_SIZE`-row subset (now 200,000 by
default), so the whole rest of the notebook finishes in well under an hour instead of the ~29
hours the full 15.2M-row run takes. This is a genuine end-to-end dry run, not a shortcut that
skips steps -- it's the same code path the full run uses, just on fewer rows. When you're ready
for the full dataset, set `SAMPLE_SIZE = None` in the config cell above and re-run from there
(the full run writes to its own `bluesky_shards_full/` dir, so it won't collide with anything
the sample run already wrote).


In [36]:
if SAMPLE_SIZE is not None and len(bluesky) > SAMPLE_SIZE:
    bluesky = bluesky.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"SAMPLE MODE: subsampled to {len(bluesky):,} rows (random_state={RANDOM_STATE}) "
          "for a quick end-to-end verification run.")
    print("Set SAMPLE_SIZE = None in the config cell above to process the full dataset.")
else:
    print(f"Running on the full dataset: {len(bluesky):,} rows.")


SAMPLE MODE: subsampled to 200,000 rows (random_state=42) for a quick end-to-end verification run.
Set SAMPLE_SIZE = None in the config cell above to process the full dataset.


### Locating the text and id columns

Not hardcoding a column name here -- checks what's actually there and picks from a short list of
likely candidates, printing the choice rather than assuming it silently.


In [37]:
TEXT_COL_CANDIDATES = ["clean_text", "cleaned_text", "clean_post_text", "text", "post_text", "content"]
ID_COL_CANDIDATES = ["id", "uri", "post_id", "post_uri", "cid"]

text_col = next((c for c in TEXT_COL_CANDIDATES if c in bluesky.columns), None)
if text_col is None:
    raise ValueError(
        f"No known text column found among {TEXT_COL_CANDIDATES}. "
        f"Available columns: {bluesky.columns.tolist()}. "
        "Add the actual column name to TEXT_COL_CANDIDATES above and re-run."
    )

id_col = next((c for c in ID_COL_CANDIDATES if c in bluesky.columns), None)
if id_col is None:
    print("No known id column found -- falling back to the row index as the post identifier.")
    bluesky = bluesky.reset_index(drop=False).rename(columns={"index": "row_id"})
    id_col = "row_id"

print(f"Using text column: '{text_col}'")
print(f"Using id column  : '{id_col}'")


Using text column: 'clean_text'
Using id column  : 'post_id'


In [38]:
import unicodedata

CLEANING_LIBS_OK = (
    ensure_package("ftfy")
    and ensure_package("emoji")
    and ensure_package("cleantext", pip_name="clean-text")
)

if CLEANING_LIBS_OK:
    import ftfy
    import emoji
    from cleantext import clean as cleantext_clean

    URL_RE = re.compile(r"http\S+|www\.\S+")

    def clean_text_nb03(text):
        '''Exact reproduction of notebook 03's clean_text().'''
        if pd.isna(text):
            return ""
        text = ftfy.fix_text(str(text))
        text = emoji.demojize(text)
        text = URL_RE.sub(" ", text)
        text = cleantext_clean(
            text,
            lower=True,
            no_urls=True,
            no_emails=True,
            no_phone_numbers=True,
            no_currency_symbols=True,
            replace_with_url="",
        )
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    if "text" in bluesky.columns and "clean_text" in bluesky.columns:
        check_n = min(2000, len(bluesky))
        check_sample = bluesky[["text", "clean_text"]].dropna().sample(check_n, random_state=42)
        recomputed = check_sample["text"].progress_apply(clean_text_nb03)
        match_rate = (recomputed.values == check_sample["clean_text"].values).mean()
        print(f"Recomputed notebook-03 clean_text() on {check_n:,} sampled rows and compared "
              f"against the stored 'clean_text' column.")
        print(f"Exact-match rate: {match_rate:.2%}")
        if match_rate > 0.98:
            print("--> CONFIRMED: Bluesky's clean_text was produced by notebook 03's cleaning "
                  "pipeline (or something functionally identical to it). The precision/recall "
                  "numbers carried over from notebook 05 apply as-is.")
        else:
            print("--> MISMATCH: stored clean_text does not reliably match notebook 03's "
                  "function on this sample. Treat keyword precision/recall as unverified for "
                  "Bluesky and inspect a few mismatched rows before trusting them.")
            mismatched = check_sample[recomputed.values != check_sample["clean_text"].values]
            print("\nExample mismatches:")
            print(mismatched.head(5))
    else:
        print("Columns 'text' and/or 'clean_text' not both present -- cannot verify directly. "
              "Falling back to a manual eyeball spot-check instead.")
        sample_n = min(5, len(bluesky))
        for t in bluesky[text_col].dropna().sample(sample_n, random_state=42).tolist():
            print("-", repr(t)[:200])
else:
    print("Could not install ftfy/emoji/clean-text in this environment -- skipping automated "
          "verification. Falling back to a manual eyeball spot-check against Jigsaw clean_text "
          "conventions (lowercase, no raw URLs/HTML):\n")
    sample_n = min(5, len(bluesky))
    for t in bluesky[text_col].dropna().sample(sample_n, random_state=42).tolist():
        print("-", repr(t)[:200])


Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.


  0%|          | 0/2000 [00:00<?, ?it/s]

Recomputed notebook-03 clean_text() on 2,000 sampled rows and compared against the stored 'clean_text' column.
Exact-match rate: 99.90%
--> CONFIRMED: Bluesky's clean_text was produced by notebook 03's cleaning pipeline (or something functionally identical to it). The precision/recall numbers carried over from notebook 05 apply as-is.


In [39]:
SHORT_TEXT_THRESHOLD = 5  # chars -- collapses like "lol", "👀", empty strings

is_dupe = bluesky[text_col].duplicated(keep=False) & bluesky[text_col].notna() & (bluesky[text_col] != "")
dupe_count = bluesky[text_col].duplicated().sum()  # first-occurrence-excluded count, for the headline number

print(f"Duplicate '{text_col}' values in Bluesky data: {dupe_count:,} out of {len(bluesky):,} "
      f"({dupe_count / len(bluesky):.2%})")

if is_dupe.any():
    dupe_rows = bluesky.loc[is_dupe, text_col]
    dupe_lengths = dupe_rows.str.len()

    short_share = (dupe_lengths <= SHORT_TEXT_THRESHOLD).mean()
    print(f"\nOf the rows involved in a duplicate group:")
    print(f"  - median length: {dupe_lengths.median():.0f} chars")
    print(f"  - share at or under {SHORT_TEXT_THRESHOLD} chars (short/emoji-style collisions): "
          f"{short_share:.1%}")

    top_dupes = (
        dupe_rows.value_counts().head(10).rename("count").reset_index()
        .rename(columns={"index": text_col})
    )
    print("\nMost frequent duplicated values:")
    print(top_dupes.to_string(index=False))

    if short_share > 0.6:
        print(
            "\n--> READS AS: mostly short/near-empty text collisions (many distinct posts "
            "happen to clean down to the same short string), not bot/repost spam. Leaving as-is "
            "is reasonable -- these aren't inflating any one group disproportionately unless "
            "short throwaway posts skew toward a particular keyword bucket, which the group "
            "table below will surface if it's happening."
        )
    else:
        print(
            "\n--> READS AS: a meaningful share of duplicates are longer, non-trivial text -- "
            "consistent with reposted/bot content rather than benign short-text collisions. "
            "Not deduplicated automatically here (that's a judgment call for the analysis "
            "phase), but this is real signal that group means below could be skewed by a small "
            "number of repeated posts. `is_duplicate_text` is carried into the final output "
            "so it can be filtered on downstream without re-deriving it."
        )
else:
    print("No duplicate text found in this run.")

bluesky["is_duplicate_text"] = is_dupe.astype(int)


Duplicate 'clean_text' values in Bluesky data: 26,015 out of 200,000 (13.01%)

Of the rows involved in a duplicate group:
  - median length: 2 chars
  - share at or under 5 chars (short/emoji-style collisions): 90.2%

Most frequent duplicated values:
clean_text  count
         !   1160
         ?    741
        ()    609
         .    438
        !!    389
       ...    222
         1    215
       !!!    211
         #    181
         2    170

--> READS AS: mostly short/near-empty text collisions (many distinct posts happen to clean down to the same short string), not bot/repost spam. Leaving as-is is reasonable -- these aren't inflating any one group disproportionately unless short throwaway posts skew toward a particular keyword bucket, which the group table below will surface if it's happening.


## Keyword Flags (Independent of the Model)

Applied exactly as in notebook 05 -- same `build_pattern()`/`load_keywords()` helpers, same
word-boundary case-insensitive regex, computed purely from text with no involvement from the
toxicity model.


In [40]:
t0 = time.time()
bluesky["gender_related_kw"] = bluesky[text_col].str.contains(gender_pattern, regex=True, na=False).astype(int)
bluesky["race_related_kw"] = bluesky[text_col].str.contains(race_pattern, regex=True, na=False).astype(int)
print(f"Keyword matching done in {time.time() - t0:.1f}s")

print("\nGender-related (keyword-based):")
print(bluesky["gender_related_kw"].value_counts())
print("\nRace-related (keyword-based):")
print(bluesky["race_related_kw"].value_counts())


Keyword matching done in 5.9s

Gender-related (keyword-based):
gender_related_kw
0    192226
1      7774
Name: count, dtype: int64

Race-related (keyword-based):
race_related_kw
0    197362
1      2638
Name: count, dtype: int64


## Load Trained Model & Tokenizer

Same runtime check as notebook 05 -- confirms which tokenizer files actually exist in
`best_model/` before assuming fast vs. slow, rather than hardcoding `use_fast`.


In [41]:
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"MODEL_DIR not found: {MODEL_DIR}. Update the path in the config cell above.")

present = {f.name for f in MODEL_DIR.glob("*")}
fast_files = [f for f in ["tokenizer.json"] if f in present]
slow_files = [f for f in ["spm.model", "vocab.txt", "merges.txt"] if f in present]

print("Files in best_model/:", sorted(present))
print("Fast-tokenizer files present:", fast_files or "NONE")
print("Slow-tokenizer files present:", slow_files or "NONE")

USE_FAST_TOKENIZER = bool(fast_files) and not slow_files
print("\n--> Loading with use_fast =", USE_FAST_TOKENIZER)


Files in best_model/: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
Fast-tokenizer files present: ['tokenizer.json']
Slow-tokenizer files present: NONE

--> Loading with use_fast = True


In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=USE_FAST_TOKENIZER)
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

print("Tokenizer type:", type(tokenizer).__name__)
print("Model loaded on:", device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Tokenizer type: DebertaV2Tokenizer
Model loaded on: cuda


In [43]:
def predict_toxicity(texts, batch_size=INFERENCE_BATCH_SIZE, max_length=MAX_LENGTH):
    probs = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Inference batches", leave=False):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            list(batch),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**encoded).logits

        batch_probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        probs.extend(batch_probs.tolist())

    return np.array(probs)

print("predict_toxicity() defined.")


predict_toxicity() defined.


## Language Detection (new)

`social_processed.parquet` has no language column — notebook 03 only carried
`post_id, user_id, instance, date, text, sent_label, sent_score` (+ derived `clean_text`) through
from the raw merged data, dropping the raw `langs` field. But notebook 02's language breakdown on
the raw 19.5M-row merge shows only **~56.4%** (`10,994,963 / 19,508,046`) of posts are tagged
`eng`; the rest is Japanese, German, Spanish, French, and dozens of others. Since both the
keyword lists and the DeBERTa model are English-only, that's ~44% of posts where a "neither"
result could just mean "not English" rather than "genuinely not gender/race-related or toxic."

Rather than leave that as a caveat, this notebook detects language itself using `langdetect`
(pure-Python, no model download required — reasonable for a checkpointed batch job). It's slower
than the keyword regex (roughly 1-2K posts/sec on one core) but far faster than the transformer
inference it runs alongside, so it's folded into the same chunked/checkpointed loop below rather
than adding a separate multi-hour pass. Detection failures (empty/whitespace/emoji-only text) are
caught and labeled `"unknown"` rather than raising.


In [44]:
LANGDETECT_OK = False
if RUN_LANGUAGE_DETECTION:
    LANGDETECT_OK = ensure_package("langdetect")

if LANGDETECT_OK:
    from langdetect import detect, DetectorFactory, LangDetectException
    DetectorFactory.seed = 42  # deterministic detection

    def detect_lang_safe(text):
        if not isinstance(text, str) or len(text.strip()) < 3:
            return "unknown"
        try:
            return detect(text)
        except LangDetectException:
            return "unknown"

    print("langdetect ready. Language detection will run inside the chunked inference loop.")
elif RUN_LANGUAGE_DETECTION:
    print("langdetect could not be installed in this environment -- language detection will be "
          "skipped even though RUN_LANGUAGE_DETECTION=True. 'detected_lang' will be filled with "
          "'unknown' and the English-only comparison will be unavailable; the all-languages "
          "comparison is unaffected.")

    def detect_lang_safe(text):
        return "unknown"
else:
    print("RUN_LANGUAGE_DETECTION=False -- skipping language detection. "
          "'detected_lang' will be filled with 'unknown'.")

    def detect_lang_safe(text):
        return "unknown"


langdetect ready. Language detection will run inside the chunked inference loop.


## Chunked Inference with Checkpointing

Bluesky is far larger than Jigsaw's 20K-row validation set, so this processes `CHUNK_SIZE` rows
at a time and writes each finished chunk to its own shard parquet under `bluesky_shards_*/`.
Shards that already exist on disk are skipped -- so re-running this cell after an interruption
(kernel restart, timeout, etc.) resumes instead of redoing completed work. Toxicity inference and
language detection both run per chunk and land in the same shard, so a resume never leaves the
two out of sync. Two progress bars: an outer one over chunks, an inner one (from
`predict_toxicity`) over inference batches within the current chunk.


In [45]:
keep_cols = [id_col, text_col, "gender_related_kw", "race_related_kw", "is_duplicate_text"]

n_rows = len(bluesky)
n_chunks = math.ceil(n_rows / CHUNK_SIZE)
already_done = len(list(SHARD_DIR.glob("shard_*.parquet")))

print(f"Processing {n_rows:,} Bluesky posts in {n_chunks} chunk(s) of up to {CHUNK_SIZE:,} rows each.")
print(f"Shards already present (will be skipped): {already_done}/{n_chunks}")
print(f"Checkpoint directory: {SHARD_DIR}\n")

chunk_times = []
remaining_after_start = n_chunks - already_done

for chunk_idx in tqdm(range(n_chunks), desc="Chunks"):
    shard_path = SHARD_DIR / f"shard_{chunk_idx:05d}.parquet"
    if shard_path.exists():
        continue

    t_chunk0 = time.time()
    start, end = chunk_idx * CHUNK_SIZE, min((chunk_idx + 1) * CHUNK_SIZE, n_rows)
    chunk_df = bluesky.iloc[start:end][keep_cols].copy()

    texts = chunk_df[text_col].fillna("").tolist()
    chunk_df["pred_prob"] = predict_toxicity(texts)
    chunk_df["pred_label"] = (chunk_df["pred_prob"] >= 0.5).astype(int)

    chunk_df["detected_lang"] = chunk_df[text_col].progress_apply(detect_lang_safe)
    chunk_df["is_english"] = (chunk_df["detected_lang"] == "en").astype(int)

    chunk_df.to_parquet(shard_path, index=False)

    elapsed = time.time() - t_chunk0
    chunk_times.append(elapsed)
    avg_chunk_time = sum(chunk_times) / len(chunk_times)
    chunks_left = remaining_after_start - len(chunk_times)
    eta_min = avg_chunk_time * chunks_left / 60

    print(f"  chunk {chunk_idx + 1}/{n_chunks}  rows [{start}:{end})  -> {shard_path.name}  "
          f"({elapsed:.1f}s, avg {avg_chunk_time:.1f}s/chunk, ETA remaining: {eta_min:.1f} min)")

print("\nAll chunks processed.")
if RUN_TAG.startswith("sample") and chunk_times:
    avg_chunk_time = sum(chunk_times) / len(chunk_times)
    est_full_chunks = math.ceil(15_223_017 / CHUNK_SIZE)
    est_full_hours = avg_chunk_time * est_full_chunks / 3600
    print(
        f"This was a SAMPLE run. At the observed {avg_chunk_time:.1f}s/chunk, the full "
        f"15,223,017-row dataset ({est_full_chunks} chunks) would take roughly "
        f"{est_full_hours:.1f} hours. Set SAMPLE_SIZE = None in the config cell once you're "
        f"happy with the results below, and plan to run it as a background job with the "
        f"checkpointing already in place."
    )


Processing 200,000 Bluesky posts in 4 chunk(s) of up to 50,000 rows each.
Shards already present (will be skipped): 0/4
Checkpoint directory: /kaggle/working/bluesky_shards_sample200000



Chunks:   0%|          | 0/4 [00:00<?, ?it/s]

Inference batches:   0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/50000 [00:00<?, ?it/s]

  chunk 1/4  rows [0:50000)  -> shard_00000.parquet  (483.6s, avg 483.6s/chunk, ETA remaining: 24.2 min)


...

  0%|          | 0/50000 [00:00<?, ?it/s]

  chunk 3/4  rows [100000:150000)  -> shard_00002.parquet  (484.6s, avg 485.4s/chunk, ETA remaining: 8.1 min)


Inference batches:   0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/50000 [00:00<?, ?it/s]

  chunk 4/4  rows [150000:200000)  -> shard_00003.parquet  (479.1s, avg 483.8s/chunk, ETA remaining: 0.0 min)

All chunks processed.
This was a SAMPLE run. At the observed 483.8s/chunk, the full 15,223,017-row dataset (305 chunks) would take roughly 41.0 hours. Set SAMPLE_SIZE = None in the config cell once you're happy with the results below, and plan to run it as a background job with the checkpointing already in place.


## Combine Shards & Compute Prior-Corrected Toxicity

### Prior-corrected toxicity probability (new)

`pred_prob` is the model's raw softmax output. It's not wrong, but it's not a real-world
probability either: the model was trained on a **class-balanced 50,000/50,000 split** (confirmed
in notebooks 03 & 04), while Jigsaw's actual toxic rate is **7.9969%** (confirmed in notebook 01).
A model trained on an artificially balanced 50/50 sample systematically overestimates the
positive-class probability relative to the true population rate — a well-known effect of training
on resampled/case-control data (the same correction used for rare-events logistic regression,
King & Zeng 2001).

The fix is a closed-form logit shift, applied after the fact to `pred_prob` — no retraining
needed:

```
adjusted_logit = logit(pred_prob) + log( (true_rate / (1 - true_rate)) / (train_rate / (1 - train_rate)) )
pred_prob_calibrated = sigmoid(adjusted_logit)
```

**This still carries an assumption**, stated explicitly rather than buried: it corrects for the
*known* training-vs-population gap, using Jigsaw's true rate as the reference population. It does
**not** know Bluesky's actual toxic rate (there's no ground truth for that), so
`pred_prob_calibrated` should be read as "what the score would be if Bluesky's underlying toxic
rate resembles Jigsaw's" — a real improvement over the raw, uncorrected number, not a claim of
perfect calibration on Bluesky specifically. Both columns are kept in the output so this
assumption is easy to interrogate rather than hidden.


In [46]:
shard_files = sorted(SHARD_DIR.glob("shard_*.parquet"))
print(f"Found {len(shard_files)} shard(s) to combine.")

bluesky_result = pd.concat(
    [pd.read_parquet(f) for f in tqdm(shard_files, desc="Loading shards")],
    ignore_index=True,
)
print("Combined shape:", bluesky_result.shape)


def calibrate_prob(p, train_pos_rate=TRAIN_POS_RATE, true_pos_rate=TRUE_TOXIC_RATE):
    '''Prior-correct a probability trained on a resampled class balance back toward a
    reference population rate, via a logit shift. Vectorized, safe at 15M+ rows.'''
    eps = 1e-6
    p = np.clip(p, eps, 1 - eps)
    logit = np.log(p / (1 - p))
    correction = np.log(
        (true_pos_rate / (1 - true_pos_rate)) / (train_pos_rate / (1 - train_pos_rate))
    )
    adjusted_logit = logit + correction
    return 1 / (1 + np.exp(-adjusted_logit))


bluesky_result["pred_prob_calibrated"] = calibrate_prob(bluesky_result["pred_prob"].values)

print(f"\nRaw pred_prob        -- mean: {bluesky_result['pred_prob'].mean():.4f}, "
      f"median: {bluesky_result['pred_prob'].median():.4f}")
print(f"Calibrated pred_prob -- mean: {bluesky_result['pred_prob_calibrated'].mean():.4f}, "
      f"median: {bluesky_result['pred_prob_calibrated'].median():.4f}")

if LANGDETECT_OK:
    lang_counts = bluesky_result["detected_lang"].value_counts()
    english_share = (bluesky_result["detected_lang"] == "en").mean()
    print(f"\nDetected language breakdown (top 10):")
    print(lang_counts.head(10))
    print(f"\nEnglish share of this run: {english_share:.1%} "
          f"(raw Bluesky merge overall was ~56.4% English per notebook 02)")

bluesky_result.head(5)


Found 4 shard(s) to combine.


Loading shards:   0%|          | 0/4 [00:00<?, ?it/s]

Combined shape: (200000, 9)

Raw pred_prob        -- mean: 0.1044, median: 0.0007
Calibrated pred_prob -- mean: 0.0789, median: 0.0001

Detected language breakdown (top 10):
detected_lang
en         102957
unknown     35076
de          18809
fr           5829
pt           4978
es           4012
nl           3482
no           2423
af           2187
so           1778
Name: count, dtype: int64

English share of this run: 51.5% (raw Bluesky merge overall was ~56.4% English per notebook 02)


,post_id,clean_text,gender_related_kw,race_related_kw,is_duplicate_text,pred_prob,pred_label,detected_lang,is_english,pred_prob_calibrated
0,28560137,"why can't we have songs that everyone can sing along to anymore, like the ones from my youth? come on, everyone. 🎶 i'm horny, horny, horny, horny.",0,0,0,0.994854,1,en,1,0.943829
1,35999961,,0,0,0,0.000526,0,unknown,0,0.000046
2,89493092,"it's #portfolioday! i'm jo, currently painting for magic, and other ttrpg stuffs.",0,0,0,0.000668,0,en,1,0.000058
3,67263581,they're good company. although i am liable to fall asleep on their kitchen table any moment now.,0,0,0,0.000660,0,en,1,0.000057
4,181656913,took me way too long to figure out who cassandra was fighting. i had the game on ps2 so we had heihachi because weapons are for chumps.,0,0,0,0.970662,1,en,1,0.741987


## Comparison: Keyword-Based Grouping vs. the No-Keyword Baseline

This is the requested comparison for Bluesky: what mean toxicity looks like **without** any
gender/race keyword inference at all (a single pooled average across every post) versus what it
looks like **with** keyword-based grouping (split into gender-only / race-only / both / neither).
The `diff_vs_overall` column is the size of that gap per group.

Four views are shown, in increasing order of how much correction has been applied:

1. **All languages, raw `pred_prob`** — closest to the previous version's output, for continuity.
2. **All languages, calibrated `pred_prob_calibrated`** — same rows, prior-corrected probability.
3. **English-only, raw `pred_prob`** — restricts to posts `langdetect` tagged `en`, so the model
   and keyword lists are actually operating in-domain.
4. **English-only, calibrated** — both fixes applied together; this is the most defensible single
   number if you need to pick one.

If (1) and (3) tell noticeably different stories, that's the ~44% non-English share doing exactly
what the caveat predicted — worth flagging explicitly rather than only trusting the pooled number.


In [47]:
def group_summary(df, gender_col, race_col, prob_col, label):
    # Vectorized bucketing (np.select), not a row-wise .apply() -- .apply(axis=1) is fine at
    # sample-run sizes but crawls at 15M+ rows, so this is written to hold up at full scale too.
    conditions = [
        (df[gender_col] == 1) & (df[race_col] == 1),
        (df[gender_col] == 1) & (df[race_col] == 0),
        (df[gender_col] == 0) & (df[race_col] == 1),
    ]
    choices = ["gender_and_race", "gender_only", "race_only"]
    group = np.select(conditions, choices, default="neither")

    overall_mean = df[prob_col].mean()
    grouped = df.assign(group=group).groupby("group")[prob_col].agg(["mean", "count"])
    grouped["diff_vs_overall"] = grouped["mean"] - overall_mean

    print(f"--- {label} ---")
    print(f"Overall (no-keyword baseline) mean: {overall_mean:.4f}   n={len(df):,}")
    print(grouped)
    print()
    return grouped


all_lang_df = bluesky_result
english_only_df = (
    bluesky_result[bluesky_result["detected_lang"] == "en"]
    if LANGDETECT_OK else None
)

summary_all_raw = group_summary(
    all_lang_df, "gender_related_kw", "race_related_kw", "pred_prob",
    "ALL LANGUAGES -- raw pred_prob"
)
summary_all_calibrated = group_summary(
    all_lang_df, "gender_related_kw", "race_related_kw", "pred_prob_calibrated",
    "ALL LANGUAGES -- calibrated pred_prob_calibrated"
)

if LANGDETECT_OK and len(english_only_df):
    summary_en_raw = group_summary(
        english_only_df, "gender_related_kw", "race_related_kw", "pred_prob",
        f"ENGLISH-ONLY ({len(english_only_df):,} posts) -- raw pred_prob"
    )
    summary_en_calibrated = group_summary(
        english_only_df, "gender_related_kw", "race_related_kw", "pred_prob_calibrated",
        f"ENGLISH-ONLY ({len(english_only_df):,} posts) -- calibrated pred_prob_calibrated"
    )
else:
    print("English-only comparison unavailable (language detection was off or found no "
          "'en'-tagged posts in this run).")
    summary_en_raw = summary_en_calibrated = None


--- ALL LANGUAGES -- raw pred_prob ---
Overall (no-keyword baseline) mean: 0.1044   n=200,000
                     mean   count  diff_vs_overall
group                                             
gender_and_race  0.534811     335         0.430458
gender_only      0.242908    7439         0.138556
neither          0.096656  189923        -0.007696
race_only        0.228838    2303         0.124486

--- ALL LANGUAGES -- calibrated pred_prob_calibrated ---
Overall (no-keyword baseline) mean: 0.0789   n=200,000
                     mean   count  diff_vs_overall
group                                             
gender_and_race  0.415263     335         0.336366
gender_only      0.185899    7439         0.107002
neither          0.073000  189923        -0.005897
race_only        0.170619    2303         0.091722

--- ENGLISH-ONLY (102,957 posts) -- raw pred_prob ---
Overall (no-keyword baseline) mean: 0.1674   n=102,957
                     mean  count  diff_vs_overall
group                

## Save Final Deliverable

Schema matches notebook 05's output where the columns apply, plus the new columns from this
version (`detected_lang`, `is_english`, `pred_prob_calibrated`, `is_duplicate_text`), so the
visualization-only notebook that follows can use either the raw or calibrated score and either
the pooled or English-only view.


In [48]:
final_cols = [
    id_col, text_col,
    "gender_related_kw", "race_related_kw",
    "pred_prob", "pred_prob_calibrated", "pred_label",
    "detected_lang", "is_english",
    "is_duplicate_text",
]
final_cols = [c for c in final_cols if c in bluesky_result.columns]

final_result = bluesky_result[final_cols].rename(columns={id_col: "post_id", text_col: "text"})

final_result.to_parquet(FINAL_OUTPUT_PATH, index=False)

print("Saved!")
print(FINAL_OUTPUT_PATH)
print(f"Run mode: {RUN_TAG}")
print("Shape:", final_result.shape)
final_result.head(10)


Saved!
/kaggle/working/bluesky_toxicity_bias_inference_sample200000.parquet
Run mode: sample200000
Shape: (200000, 10)


,post_id,text,gender_related_kw,race_related_kw,pred_prob,pred_prob_calibrated,pred_label,detected_lang,is_english,is_duplicate_text
0,28560137,"why can't we have songs that everyone can sing along to anymore, like the ones from my youth? come on, everyone. 🎶 i'm horny, horny, horny, horny.",0,0,0.994854,0.943829,1,en,1,0
1,35999961,,0,0,0.000526,0.000046,0,unknown,0,0
2,89493092,"it's #portfolioday! i'm jo, currently painting for magic, and other ttrpg stuffs.",0,0,0.000668,0.000058,0,en,1,0
3,67263581,they're good company. although i am liable to fall asleep on their kitchen table any moment now.,0,0,0.000660,0.000057,0,en,1,0
4,181656913,took me way too long to figure out who cassandra was fighting. i had the game on ps2 so we had heihachi because weapons are for chumps.,0,0,0.970662,0.741987,1,en,1,0
5,21309504,~ ? 🌶️,0,0,0.000548,0.000048,0,unknown,0,0
6,173448065,fast man undrar vad som doljs i kalsongerna.,1,0,0.004064,0.000355,0,sv,0,0
7,66699118,( ),0,0,0.000546,0.000048,0,unknown,0,1
8,110212688,talvez mais tarde kk,0,0,0.000546,0.000047,0,hu,0,0
9,3386475,"""lee told me she has had multiple supporters say to her, ""'barbara, we love you, but adam schiff just looks like a senator '...i guess they're rig...",1,1,0.072764,0.006775,0,en,1,0
